# CFM vs standard MLP: 100 runs per method

Train independent networks using the existing standard MSE and CFM endpoint-residual implementations. Each seed is used once per method, with identical hidden width, optimizer update count, learning rate, training points, and evaluation grid. Paired seed IDs do not imply identical initial parameters: CFM has two inputs and `4*hidden_size+1` parameters, while the standard MLP has one input and `3*hidden_size+1` parameters.

The main metric is **final grid MSE**, evaluated only after training. CFM uses `eval_sampling_steps` Euler steps starting from zero noise for every x. Training MSE and training time are also saved. There are no live fit charts during the batch. The progress display reports the current network and training step.

The final Plotly box plot shows both distributions, individual runs, and highlighted **arithmetic means** on a logarithmic MSE axis. The table reports mean, sample standard deviation, median, quartiles, minimum, and maximum. Results are saved after every completed network to `notebooks/outputs/cfm_vs_standard.csv`; rerunning the training cell starts a new batch and replaces this file.

In [1]:
from pathlib import Path
import sys
import csv
import json
from dataclasses import asdict, replace
from time import perf_counter

project_root = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "pyproject.toml").is_file() and (p / "notebooks" / "_shared").is_dir()
)
shared_parent = str(project_root / "notebooks")
if shared_parent not in sys.path:
    sys.path.insert(0, shared_parent)

import numpy as np
import torch
import ipywidgets as widgets
import plotly.graph_objects as go
from IPython.display import display, Markdown
from _shared import Config, Experiment
from _shared.conditional_flow_matching import FlowMatchingConfig
from _shared.flow_matching_experiment import FlowMatchingExperiment

torch.set_num_threads(1)

In [2]:
n_runs = 100  # Per method: 200 networks in total.
config = Config(
    hidden_size=32,
    total_steps=20_000,
    learning_rate=0.01,
    seed=42,
    n_train=256,
    n_grid=601,
    x_min=-3.0,
    x_max=3.0,
)
flow_config = FlowMatchingConfig(
    eval_sampling_steps=1,
    train_sampling_steps=None,
)
methods = ("Standard MLP", "CFM")
output_dir = project_root / "notebooks" / "outputs"
results_path = output_dir / "cfm_vs_standard.csv"

In [3]:
if not isinstance(n_runs, int) or n_runs < 1:
    raise ValueError("n_runs must be a positive integer.")

results = []
output_dir.mkdir(parents=True, exist_ok=True)
metadata = {
    "n_runs_per_method": n_runs,
    "config": asdict(config),
    "flow_config": asdict(flow_config),
    "primary_metric": "grid_mse",
    "cfm_evaluation_initial_noise": 0,
    "torch_version": torch.__version__,
}
(output_dir / "cfm_vs_standard_config.json").write_text(
    json.dumps(metadata, indent=2) + "\n", encoding="utf-8"
)
columns = ["method", "seed", "train_mse", "grid_mse", "training_seconds"]
with results_path.open("w", newline="", encoding="utf-8") as stream:
    csv.DictWriter(stream, fieldnames=columns).writeheader()

completed = widgets.IntProgress(
    value=0,
    min=0,
    max=n_runs * len(methods),
    description="Networks:",
    layout=widgets.Layout(width="100%"),
)
current = widgets.HTML()
display(widgets.VBox([completed, current]))
progress_every = max(1, config.total_steps // 100)

for run_index in range(n_runs):
    run_config = replace(config, seed=config.seed + run_index)
    for method in methods:
        experiment = (
            Experiment(run_config)
            if method == "Standard MLP"
            else FlowMatchingExperiment(run_config, flow_config)
        )
        started = perf_counter()
        for progress in experiment.run():
            if (
                progress.step % progress_every == 0
                or progress.step == config.total_steps
            ):
                current.value = (
                    f"<b>{method}</b> | run {run_index + 1}/{n_runs} | seed {run_config.seed}"
                    f" | step {progress.step:,}/{config.total_steps:,}"
                )
        training_seconds = perf_counter() - started
        snapshot = experiment.evaluate_model(experiment.model, experiment.progress)
        if not np.isfinite([snapshot.train_mse, snapshot.grid_mse]).all():
            raise FloatingPointError(
                f"Nonfinite final MSE: {method}, seed {run_config.seed}."
            )
        record = {
            "method": method,
            "seed": run_config.seed,
            "train_mse": snapshot.train_mse,
            "grid_mse": snapshot.grid_mse,
            "training_seconds": training_seconds,
        }
        results.append(record)
        with results_path.open("a", newline="", encoding="utf-8") as stream:
            csv.DictWriter(stream, fieldnames=columns).writerow(record)
        completed.value = len(results)
        del experiment

completed.bar_style = "success"
current.value = f"Finished {len(results)} networks. Results saved to <code>{results_path.name}</code>."

## Final MSE distributions

The report uses the completed runs in `results`, so it also works with a partial batch after an interruption. To report a previously saved batch in a fresh kernel, run the imports and configuration cells, then run the CSV-loading cell below instead of the training cell.

In [4]:
# Load the completed results saved by the training cell.
with results_path.open(newline="", encoding="utf-8") as stream:
    results = [
        {
            "method": row["method"],
            "seed": int(row["seed"]),
            "train_mse": float(row["train_mse"]),
            "grid_mse": float(row["grid_mse"]),
            "training_seconds": float(row["training_seconds"]),
        }
        for row in csv.DictReader(stream)
    ]

In [5]:
if not results:
    raise ValueError(
        "No completed runs are available. Train models or load saved results first."
    )

summary = []
for method in methods:
    values = np.array([row["grid_mse"] for row in results if row["method"] == method])
    if not len(values):
        continue
    summary.append(
        {
            "method": method,
            "runs": len(values),
            "mean": float(values.mean()),
            "std": float(values.std(ddof=1)) if len(values) > 1 else float("nan"),
            "min": float(values.min()),
            "q25": float(np.quantile(values, 0.25)),
            "median": float(np.median(values)),
            "q75": float(np.quantile(values, 0.75)),
            "max": float(values.max()),
        }
    )

table = [
    "| Method | Runs | Mean | Sample std | Min | Q25 | Median | Q75 | Max |",
    "|:--|--:|--:|--:|--:|--:|--:|--:|--:|",
]
for row in summary:
    numbers = [
        f"{row[key]:.6g}"
        for key in ("mean", "std", "min", "q25", "median", "q75", "max")
    ]
    table.append(f"| {row['method']} | {row['runs']} | " + " | ".join(numbers) + " |")
display(Markdown("### Final grid MSE\n\n" + "\n".join(table)))

### Final grid MSE

| Method | Runs | Mean | Sample std | Min | Q25 | Median | Q75 | Max |
|:--|--:|--:|--:|--:|--:|--:|--:|--:|
| Standard MLP | 100 | 0.0195202 | 0.0195702 | 0.000295755 | 0.00288547 | 0.0077071 | 0.0304163 | 0.0664809 |
| CFM | 100 | 0.00161295 | 0.00129855 | 0.000200195 | 0.000643771 | 0.00148449 | 0.00196762 | 0.00552163 |

In [6]:
figure = go.Figure()
colors = {"Standard MLP": "#2563eb", "CFM": "#16a34a"}
plot_floor = np.finfo(float).tiny
for method in methods:
    rows = [row for row in results if row["method"] == method]
    if not rows:
        continue
    figure.add_trace(
        go.Box(
            name=method,
            x=[method] * len(rows),
            y=[max(row["grid_mse"], plot_floor) for row in rows],
            customdata=[[row["seed"], row["train_mse"]] for row in rows],
            boxpoints="all",
            jitter=0.35,
            pointpos=0,
            quartilemethod="linear",
            marker=dict(color=colors[method], size=5, opacity=0.55),
            line=dict(color=colors[method], width=2),
            showlegend=False,
            hovertemplate=(
                f"{method}<br>Grid MSE: %{{y:.6g}}<br>Seed: %{{customdata[0]}}"
                "<br>Train MSE: %{customdata[1]:.6g}<extra></extra>"
            ),
        )
    )
figure.add_trace(
    go.Scatter(
        x=[row["method"] for row in summary],
        y=[max(row["mean"], plot_floor) for row in summary],
        text=[f"Mean: {row['mean']:.4g}" for row in summary],
        mode="markers+text",
        textposition="top center",
        marker=dict(
            symbol="diamond",
            color="#f59e0b",
            size=15,
            line=dict(color="#78350f", width=2),
        ),
        name="Arithmetic mean",
        hovertemplate="%{x}<br>Mean MSE: %{y:.6g}<extra></extra>",
    )
)
figure.update_layout(
    title=dict(
        text="Final grid MSE: standard MLP vs CFM",
        x=0, xanchor="left", y=0.98, yanchor="top", font=dict(size=20),
    ),
    template="plotly_white",
    autosize=True,
    height=650,
    xaxis=dict(title="Method", categoryorder="array", categoryarray=list(methods)),
    yaxis=dict(title="Final grid MSE (lower is better)", type="log"),
    margin=dict(l=80, r=35, t=80, b=130),
    legend=dict(orientation="h", x=0.5, xanchor="center", y=-0.2, yanchor="top"),
)
figure.show(config={"responsive": True, "displayModeBar": False})